In [ ]:
import torch
from diffusers import AutoPipelineForInpainting

pipeline = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
pipeline.enable_model_cpu_offload()


def generate_patched_image(init_image, mask_image):
    init_image_downscaled = init_image.resize((1024, 1024))
    mask_image_downscaled = mask_image.resize((1024, 1024))

    # generator = torch.Generator("cuda").manual_seed(92)
    prompt = "a clean petri dish with nutrient agar medium under laboratory lighting"
    negative_prompt = (
        "bacteria colony, bacterial colonies, bacteria, circles, dots, microbial spots"
    )
    image = pipeline(
        prompt=prompt,
        strength=0.99,
        negative_prompt=negative_prompt,
        image=init_image_downscaled,
        mask_image=mask_image_downscaled,
        # num_images_per_prompt=4,
        # generator=generator,
    ).images[0]

    # Upscale the image to the original size
    image_upscaled = image.resize(init_image.size, resample=3)

    return image_upscaled

In [ ]:
from PIL import Image
import numpy as np
from pathlib import Path


def count_colonies(mask_path: Path):
    image = Image.open(mask_path)
    image = image.convert("RGB")

    image_data = np.array(image)
    # Reshape the image data to a 2D array where each row is a color
    pixels = image_data.reshape(-1, image_data.shape[-1])

    unique_colors = np.unique(pixels, axis=0)

    # Exclude black (background)
    return len(unique_colors) - 1

In [ ]:
import json
import cv2

images_path = Path("../data/labeled/images")
masks_path = Path("../data/labeled/masks")

patches_path = Path("./patched_images")
patches_path.mkdir(exist_ok=True)

with open("circles.json") as f:
    circles_data = json.load(f)

for i, (img_id, circles) in enumerate(circles_data.items()):
    colonies = count_colonies(masks_path / f"{img_id}.png")

    if colonies < 10:
        print(f"Skipping image {img_id} ({i + 1}/{len(circles_data)})")
        continue

    print(
        f"Processing image {img_id} ({i + 1}/{len(circles_data)}) with {colonies} colonies"
    )

    for j, circle in enumerate(circles):
        print(f"  Processing circle {j + 1}/{len(circles)}")

        x, y, radius = circle["x"], circle["y"], circle["r"]

        image = Image.open(images_path / f"{img_id}.jpg")

        circle_mask = np.zeros(image.size, dtype=np.uint8)
        cv2.circle(circle_mask, (x, y), radius, (255, 255, 255), -1)

        mask_image = Image.fromarray(circle_mask)

        patched_image = generate_patched_image(image, mask_image).save(
            patches_path / f"{img_id}_{x}_{y}_{radius}.jpg"
        )